In [1]:
## run as script
import os # For System operations
import sys
sys.path.append('/shared/home/v_neelesh_bisht/local_scratch/3d-cnn')

import nibabel as nib # To load .nii.gz files
import cv2 # for image operations
import matplotlib.pyplot as plt # To plot the images
import numpy as np # Numpy operations\
from tensorflow.keras.models import load_model # To load the model
import numpy as np # Numpy operations
from utils import Device

2024-07-26 03:23:08.256604: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2024-07-26 03:23:08.256693: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2024-07-26 03:23:08.327881: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2024-07-26 03:23:08.777660: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2024-07-26 03:23:10.392145: W tensorflow/compiler/tf2

In [2]:
device_obj = Device()
device_obj.set_device(6)
device = device_obj.get_device()
print(f"Using device: {device}")

Using device: cuda:6


In [3]:

HOUNSFIELD_MIN = -3500 # assigned minimum HOUNSFIELD value
HOUNSFIELD_MAX = 3500 # assigned maximum HOUNSFIELD value
HOUNSFIELD_RANGE = HOUNSFIELD_MAX - HOUNSFIELD_MIN # difference of HOUNSFIELD max and min values

# these variables define from what side/plane should be sliced
SLICE_X = True # represents the sagittal slice
SLICE_Y = True # represents the coronal slice
SLICE_Z = True # represents the axial slice

SLICE_DECIMATE_IDENTIFIER = 3

In [4]:
# Function which normalises the values
def normalizeImageIntensityRange(img):
    """
    This function normalizes the image intensity range from -3500 to 3500 to 0 to 1
    """
    img[img < HOUNSFIELD_MIN] = HOUNSFIELD_MIN
    img[img > HOUNSFIELD_MAX] = HOUNSFIELD_MAX
    return (img - HOUNSFIELD_MIN) / HOUNSFIELD_RANGE

In [5]:
def readImageVolume(imgPath, normalize=False):
    """
    This function function load the image and then if necessary normalise the image
    """
    img = nib.load(imgPath).get_fdata()
    if normalize:
        return normalizeImageIntensityRange(img)
    else:
        return img

In [6]:
# Defining Constants for training

SEED = 42 # Seed for reproducibility
IMAGE_HEIGHT = 80 # Set the image height for training
IMAGE_WIDTH = 40 # Set the image width for training
IMG_SIZE = (IMAGE_HEIGHT, IMAGE_WIDTH)

In [7]:
model_path = os.path.join('/shared/home/v_neelesh_bisht/local_scratch/3d-cnn', "models/archive/lung_seg_unet/UNET_LungSegmentation_3D_10epochs_GPU_ALL.h5")
model = load_model(model_path)

2024-07-26 03:23:16.160427: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1929] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 1039 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 3090, pci bus id: 0000:01:00.0, compute capability: 8.6
2024-07-26 03:23:16.161910: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1929] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 733 MB memory:  -> device: 1, name: NVIDIA GeForce RTX 3090, pci bus id: 0000:25:00.0, compute capability: 8.6
2024-07-26 03:23:16.163128: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1929] Created device /job:localhost/replica:0/task:0/device:GPU:2 with 1665 MB memory:  -> device: 2, name: NVIDIA GeForce RTX 3090, pci bus id: 0000:41:00.0, compute capability: 8.6
2024-07-26 03:23:16.164329: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1929] Created device /job:localhost/replica:0/task:0/device:GPU:3 with 1390 MB memory:  -> device: 3, name: NVIDIA GeForce RTX 3090, pci

In [8]:
# Scale the image accordingly
def scaleImg(img, height, width):
    return cv2.resize(img, dsize=(width, height), interpolation=cv2.INTER_LINEAR)

In [9]:
def predictVolume(inImg, toBin=True):
    (xMax, yMax, zMax) = inImg.shape

    outImgX = np.zeros((xMax, yMax, zMax)) # Create imgx in the shape of inImgx
    outImgY = np.zeros((xMax, yMax, zMax)) # Create imgy in the shape of inImgy
    outImgZ = np.zeros((xMax, yMax, zMax)) # Create imgz in the shape of inImgz

    cnt = 0.0
    if SLICE_X:
        cnt += 1.0
        for i in range(xMax):
            img = scaleImg(inImg[i,:,:], IMAGE_HEIGHT, IMAGE_WIDTH)[np.newaxis,:,:,np.newaxis]
            tmp = model.predict(img)[0,:,:,0]
            outImgX[i,:,:] = scaleImg(tmp, yMax, zMax)
    if SLICE_Y:
        cnt += 1.0
        for i in range(yMax):
            img = scaleImg(inImg[:,i,:], IMAGE_HEIGHT, IMAGE_WIDTH)[np.newaxis,:,:,np.newaxis]
            tmp = model.predict(img)[0,:,:,0]
            outImgY[:,i,:] = scaleImg(tmp, xMax, zMax)
    if SLICE_Z:
        cnt += 1.0
        for i in range(zMax):
            img = scaleImg(inImg[:,:,i], IMAGE_HEIGHT, IMAGE_WIDTH)[np.newaxis,:,:,np.newaxis]
            tmp = model.predict(img)[0,:,:,0]
            outImgZ[:,:,i] = scaleImg(tmp, xMax, yMax)

    outImg = (outImgX + outImgY + outImgZ)/cnt # Concatenate all the sides to form into one
    if(toBin):
        outImg[outImg>0.5] = 1.0 # Appying thresholding if > 0.5 then assign 1
        outImg[outImg<=0.5] = 0.0 # Appying thresholding if <= 0.5 then assign 0
    return outImg

In [10]:
output_dir = os.path.join('/shared/home/v_neelesh_bisht/local_scratch/3d-cnn', "MosMedData")
paths = [os.path.join(output_dir, "CT-0", x) for x in sorted(os.listdir(os.path.join(output_dir, "CT-0")))]
save_path = os.path.join(output_dir, "CT-0-mask")

In [11]:
# Load a sample data
for path in paths[44:]:
  imgTargetNii = nib.load(path).get_fdata() # load the random 3d image
  imgTarget = normalizeImageIntensityRange(imgTargetNii) # Normalize the 3d ct scan image
  predImg = predictVolume(imgTarget) # get the volume prediction
  # print("imgTarget.shape: ",imgTarget.shape, ", predImg.shape: ", predImg.shape)
  a = nib.Nifti1Image(predImg, affine=np.eye(4))
  last_name = os.path.basename(path)
  nib.save(a, f"{save_path}/{last_name}")

2024-07-26 03:23:17.438174: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:454] Loaded cuDNN version 8902


1/1 [==============================] - 0s 52ms/step


In [ ]:
from scipy import ndimage

path = paths[0]
last_name = os.path.basename(path)
imgTargetNii = nib.load(path).get_fdata() # load the random 3d image
imgTarget = normalizeImageIntensityRange(imgTargetNii)

# Load one of the 3d masks
output_dir = os.path.join('/shared/home/v_neelesh_bisht/local_scratch/3d-cnn', "MosMedData")
maskPath = f"{output_dir}/CT-0-mask/{last_name}"
mask = nib.load(maskPath).get_fdata()
print(f"Minimum Value : {np.min(mask)}\nMaximum Value : {np.max(mask)}\nShape : {mask.shape}\nType : {type(mask)}")


imgTarget = ndimage.rotate(imgTarget, 90, reshape=False)
mask = ndimage.rotate(mask, 90, reshape=False)


max_slice = mask.shape[2]
min_slice = 0

fig, ax = plt.subplots(max_slice - min_slice, 3, figsize=(30, 5 * (max_slice - min_slice + 1)))

for i in range(min_slice, max_slice):
    slice_idx = i - min_slice

    ax[slice_idx, 0].imshow(np.squeeze(imgTarget[:, :, i]), cmap='gray')
    ax[slice_idx, 1].imshow(np.squeeze(mask[:, :, i]), cmap='gray')

    img3 = ax[slice_idx, 2].imshow(np.squeeze(imgTarget[:, :, i]), cmap='bone')
    img4 = ax[slice_idx, 2].imshow(np.squeeze(mask[:, :, i]), cmap='jet', alpha=0.5, extent=img3.get_extent())




In [ ]:

for i in range(imgTarget.shape[2]):
  plt.imshow(np.squeeze(imgTarget[:, :, i]), cmap='gray')
  plt.show()

for i in range(mask.shape[2]):
  plt.imshow(np.squeeze(mask[:, :, i ]), cmap='gray')
  plt.show()